# IBM Natural Language Understanding — Demo avanzado

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ecamposv/nlp-vision/blob/main/semana-04/notebook/IBM_NLU_Demo.ipynb)

Puedes ejecutar este notebook localmente (VS Code / Jupyter) o en **Google Colab** dando clic en el badge de arriba.

Este cuaderno extiende la demostración básica del servicio **Natural Language Understanding (NLU)** de IBM Cloud con dos objetivos pedagógicos:

1. **Estresar al modelo** con un *corpus adversarial* — sarcasmo, doble negación, sentimiento mixto, modismos y frases sutiles donde la polaridad léxica engaña — y medir la precisión contra etiquetas humanas.
2. **Conectar con un caso real** analizando una muestra del dataset *Customer Support Tickets* que también alimenta la app **VoxCustomer** (`semana-04/voxcustomer/data/customer_support_tickets.csv`).

A lo largo del flujo harás lo siguiente:

- Configurar credenciales y crear el cliente de NLU.
- Cargar dos corpus: uno adversarial etiquetado a mano y una muestra de tickets reales.
- Ejecutar análisis de **sentimiento**, **emociones**, **palabras clave** y **categorías**.
- Calcular la **precisión por tipo de frase** (sarcasmo, negación, mixto…) con una matriz de confusión.
- Visualizar resultados con `matplotlib` + `seaborn`.

> **Credenciales obligatorias:** este cuaderno requiere credenciales válidas de IBM NLU. Si no puede conectarse al servicio, las celdas lanzarán un error en lugar de continuar.

> **En Colab:** la celda de *Setup* (sección 1) instala las dependencias **y** descarga el CSV de tickets directamente desde GitHub — no hace falta subir archivos a mano.


## 1. Preparar el entorno

Ejecuta esta celda **primero**. Detecta automáticamente si corres en Google Colab o en tu máquina:

- **Colab** → instala `ibm-watson`, `matplotlib` y `seaborn` y descarga `customer_support_tickets.csv` desde el repo de GitHub a `/content/`.
- **Local (VS Code / Jupyter)** → asume que ya tienes el entorno listo (`pip install -r requirements.txt` en `semana-04/voxcustomer/`).


In [ ]:
# === Setup para Google Colab ===
# Esta celda solo hace algo cuando el notebook se ejecuta en Colab.
# En entornos locales (VS Code / Jupyter) se ignora y puede saltarse.
import sys, os, urllib.request

IN_COLAB = "google.colab" in sys.modules

# Si publicas el repo con otro nombre/branch, ajusta estas dos constantes:
GITHUB_USER_REPO = "ecamposv/nlp-vision"
GITHUB_BRANCH    = "main"
CSV_REPO_PATH    = "semana-04/voxcustomer/data/customer_support_tickets.csv"

if IN_COLAB:
    print("Colab detectado — instalando dependencias…")
    !pip install -q ibm-watson pandas matplotlib seaborn

    csv_url   = f"https://raw.githubusercontent.com/{GITHUB_USER_REPO}/{GITHUB_BRANCH}/{CSV_REPO_PATH}"
    csv_local = "/content/customer_support_tickets.csv"
    if not os.path.exists(csv_local):
        print(f"Descargando dataset de tickets desde GitHub…\n  {csv_url}")
        try:
            urllib.request.urlretrieve(csv_url, csv_local)
            print(f"  ✓ Guardado en {csv_local} ({os.path.getsize(csv_local) / 1024:.0f} KB)")
        except Exception as exc:
            print(f"  ✗ No se pudo descargar: {exc}")
            print("    Súbelo manualmente desde el panel de archivos de Colab.")
    else:
        print(f"Dataset ya presente en {csv_local}, no se descarga de nuevo.")
    print("Setup de Colab completado.")
else:
    print("Entorno local detectado, no se requiere setup adicional.")


## 2. Importar dependencias y utilidades
Este bloque importa las librerías que usaremos durante el análisis.

In [ ]:
import json
import os
from getpass import getpass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ibm_watson import NaturalLanguageUnderstandingV1
from ibm_watson.natural_language_understanding_v1 import (
    CategoriesOptions,
    EmotionOptions,
    Features,
    KeywordsOptions,
    SentimentOptions,
)
from ibm_cloud_sdk_core.authenticators import IAMAuthenticator

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams.update({
    'figure.dpi': 110,
    'axes.titleweight': 'semibold',
    'axes.spines.top': False,
    'axes.spines.right': False,
})
pd.set_option('display.max_colwidth', 90)


## 3. Configurar credenciales de IBM Cloud
Para mantener seguras tus llaves, este patrón las solicita de forma interactiva y las guarda en variables de entorno temporales dentro del entorno de ejecución.

In [ ]:
def set_env_if_missing(var_name: str, prompt: str) -> None:
    # Solicita un valor si la variable de entorno no está definida.
    if not os.environ.get(var_name):
        value = getpass(prompt)
        if value:
            os.environ[var_name] = value

set_env_if_missing('IBM_NLU_APIKEY', 'Introduce tu API Key de IBM NLU: ')
set_env_if_missing('IBM_NLU_URL', 'Introduce la URL del servicio NLU: ')

print('Credenciales cargadas (si se proporcionaron).')

## 4. Crear el cliente de Natural Language Understanding
Esta función centraliza la creación del cliente de NLU y maneja el caso en que las credenciales no estén disponibles.

In [ ]:
def build_nlu_client(version: str = '2023-09-01') -> NaturalLanguageUnderstandingV1:
    api_key = os.environ.get('IBM_NLU_APIKEY')
    service_url = os.environ.get('IBM_NLU_URL')

    if not api_key or not service_url:
        raise RuntimeError(
            'Faltan credenciales de IBM NLU. Define IBM_NLU_APIKEY y IBM_NLU_URL '
            'antes de continuar (ejecuta la celda anterior).'
        )

    authenticator = IAMAuthenticator(api_key)
    client = NaturalLanguageUnderstandingV1(version=version, authenticator=authenticator)
    client.set_service_url(service_url)
    print('Cliente de NLU configurado correctamente.')
    return client

nlu_client = build_nlu_client()

## 5. Corpus adversarial — frases complejas para retar al modelo

Vamos a ir más allá del *happy path*. Estas frases mezclan **sarcasmo**, **negación**, **doble negación**, **sentimiento mixto**, **modismos** y casos **sutiles** donde la polaridad real no coincide con las palabras "positivas" o "negativas" que contiene el texto. Cada frase incluye una etiqueta humana (`expected_sentiment`) que usaremos más adelante para calcular precisión.


In [ ]:
# Corpus adversarial: textos diseñados para que la polaridad léxica no
# coincida con la intención real. Etiquetas hechas a mano.
TRICKY_DOCUMENTS = [
    {'id': 'sarcasm_01', 'category': 'sarcasm', 'expected_sentiment': 'negative', 'language': 'en',
     'text': 'Oh fantastic — only three weeks to get a reply from your support team. Exactly what every paying customer dreams of.'},
    {'id': 'sarcasm_02', 'category': 'sarcasm', 'expected_sentiment': 'negative', 'language': 'en',
     'text': 'Brilliant work shipping me the wrong replacement adapter twice in a row. Truly an inspiring level of attention to detail.'},
    {'id': 'negation_01', 'category': 'negation', 'expected_sentiment': 'positive', 'language': 'en',
     'text': "I cannot honestly say I've had a single bad experience with this product across an entire year of daily use."},
    {'id': 'negation_02', 'category': 'negation', 'expected_sentiment': 'negative', 'language': 'en',
     'text': "It's not that the agent wasn't trying — it's that nothing they did actually moved the ticket forward."},
    {'id': 'mixed_01', 'category': 'mixed', 'expected_sentiment': 'negative', 'language': 'en',
     'text': 'The hardware is beautifully designed and the unboxing felt premium, but the device crashes roughly every twenty minutes of use.'},
    {'id': 'mixed_02', 'category': 'mixed', 'expected_sentiment': 'positive', 'language': 'en',
     'text': 'Setup was painful and the manual was nearly useless; once it was running, however, the product replaced three of our previous tools.'},
    {'id': 'subtle_01', 'category': 'subtle', 'expected_sentiment': 'positive', 'language': 'en',
     'text': 'After the last two firmware updates the freezing issue has not reappeared, and I am cautiously optimistic that this is finally stable.'},
    {'id': 'subtle_02', 'category': 'subtle', 'expected_sentiment': 'negative', 'language': 'en',
     'text': 'The agent followed every line of the script, asked the standard questions, and then closed my ticket without resolving anything.'},
    {'id': 'idiom_01', 'category': 'idiom', 'expected_sentiment': 'negative', 'language': 'en',
     'text': 'Honestly, dealing with your billing department feels like pulling teeth — I am about to start shopping for an alternative provider.'},
    {'id': 'idiom_02', 'category': 'idiom', 'expected_sentiment': 'positive', 'language': 'en',
     'text': 'Your engineer went the extra mile, stayed on the call past midnight and made sure every single service was back up before hanging up.'},
    {'id': 'es_sarcasm', 'category': 'sarcasm', 'expected_sentiment': 'negative', 'language': 'es',
     'text': 'Excelente, ahora resulta que la garantía no cubre exactamente la pieza que falló a las dos semanas. Mis más sinceras felicitaciones al equipo de producto.'},
    {'id': 'es_negation', 'category': 'negation', 'expected_sentiment': 'negative', 'language': 'es',
     'text': 'No es que el producto sea malo en sí, simplemente no hace ni la mitad de lo que prometía la página de la tienda en línea.'},
]

tricky_df = pd.DataFrame(TRICKY_DOCUMENTS)
print(f'Corpus adversarial: {len(tricky_df)} frases · {tricky_df["category"].nunique()} categorías · idiomas: {sorted(tricky_df["language"].unique())}')
tricky_df[['id', 'category', 'language', 'expected_sentiment', 'text']]


## 6. Ejecutar un análisis

`analyze_text` encapsula la llamada al servicio de IBM NLU. Nota: el análisis de **emoción** de NLU solo soporta inglés, así que lo activamos condicionalmente según el idioma del documento. Si la llamada falla, se propaga la excepción para que el problema sea evidente.


In [ ]:
def analyze_text(text: str,
                 language: str = 'en',
                 client: NaturalLanguageUnderstandingV1 | None = None) -> dict:
    if client is None:
        raise RuntimeError('Cliente de NLU no inicializado. Revisa las credenciales.')

    # Emotion solo está disponible en inglés; el resto es multilingüe.
    feature_kwargs = {
        'sentiment': SentimentOptions(document=True),
        'keywords': KeywordsOptions(limit=5, sentiment=True),
        'categories': CategoriesOptions(limit=3),
    }
    if language == 'en':
        feature_kwargs['emotion'] = EmotionOptions(document=True)

    response = client.analyze(
        text=text,
        language=language,
        features=Features(**feature_kwargs),
    ).get_result()
    return response


In [ ]:
first_doc = TRICKY_DOCUMENTS[0]  # ejemplo sarcástico
analysis_result = analyze_text(first_doc['text'], language=first_doc['language'], client=nlu_client)

print(f"Texto:    {first_doc['text']}")
print(f"Esperado: {first_doc['expected_sentiment']}")
print(f"Predicho: {analysis_result.get('sentiment', {}).get('document', {}).get('label', 'n/a')}\n")
print(json.dumps(analysis_result, indent=2)[:1500] + '\n…(salida recortada)')


## 7. Convertir la respuesta en estructuras tabulares
Estas utilidades nos permiten extraer la información clave en formatos amigables: tablas de sentimientos, emociones y palabras clave.

In [ ]:
def summarize_sentiment(response: dict) -> pd.DataFrame:
    document = response.get('sentiment', {}).get('document', {})
    if not document:
        return pd.DataFrame()
    return pd.DataFrame([
        {
            'label': document.get('label'),
            'score': document.get('score'),
        }
    ])


def summarize_emotions(response: dict) -> pd.DataFrame:
    emotions = response.get('emotion', {}).get('document', {}).get('emotion', {})
    if not emotions:
        return pd.DataFrame()
    return (
        pd.Series(emotions, name='score')
        .rename_axis('emotion')
        .reset_index()
        .sort_values('score', ascending=False)
    )


def summarize_keywords(response: dict) -> pd.DataFrame:
    keywords = response.get('keywords', [])
    if not keywords:
        return pd.DataFrame()
    rows = []
    for kw in keywords:
        rows.append(
            {
                'keyword': kw.get('text'),
                'relevance': kw.get('relevance'),
                'sentiment': kw.get('sentiment', {}).get('score'),
            }
        )
    return pd.DataFrame(rows)


def summarize_categories(response: dict) -> pd.DataFrame:
    categories = response.get('categories', [])
    if not categories:
        return pd.DataFrame()
    return pd.DataFrame(categories)

sentiment_df = summarize_sentiment(analysis_result)
emotions_df = summarize_emotions(analysis_result)
keywords_df = summarize_keywords(analysis_result)
categories_df = summarize_categories(analysis_result)

sentiment_df, emotions_df, keywords_df, categories_df

## 8. Analizar el corpus adversarial en batch

Ejecuta `analyze_text` sobre cada fila de `tricky_df` y compara la predicción contra la etiqueta humana. La columna `correct` es el indicador atómico que después agregaremos por categoría.


In [ ]:
def analyze_dataset(dataframe: pd.DataFrame,
                    client: NaturalLanguageUnderstandingV1 | None = None,
                    expected_column: str | None = 'expected_sentiment') -> pd.DataFrame:
    records = []
    for row in dataframe.to_dict(orient='records'):
        result = analyze_text(row['text'], language=row.get('language', 'en'), client=client)
        sentiment = result.get('sentiment', {}).get('document', {})
        emotion_map = result.get('emotion', {}).get('document', {}).get('emotion', {})
        dominant = max(emotion_map.items(), key=lambda kv: kv[1], default=('n/a', None))

        record = {
            'id': row.get('id'),
            'category': row.get('category', row.get('source', 'general')),
            'language': row.get('language', 'en'),
            'sentiment_label': sentiment.get('label'),
            'sentiment_score': sentiment.get('score'),
            'dominant_emotion': dominant[0],
            'emotion_score': dominant[1],
            'snippet': (row['text'][:75] + '…') if len(row['text']) > 75 else row['text'],
        }
        if expected_column and expected_column in row:
            record['expected'] = row[expected_column]
            record['correct'] = sentiment.get('label') == row[expected_column]
        records.append(record)
    return pd.DataFrame(records)


tricky_summary = analyze_dataset(tricky_df, client=nlu_client)

# Estiliza para resaltar aciertos/errores
def _highlight(row):
    if 'correct' not in row.index or pd.isna(row['correct']):
        return [''] * len(row)
    color = 'background-color: #d4edda' if row['correct'] else 'background-color: #f8d7da'
    return [color] * len(row)

display_cols = ['id', 'category', 'language', 'expected', 'sentiment_label',
                'sentiment_score', 'dominant_emotion', 'snippet']
display_cols = [c for c in display_cols if c in tricky_summary.columns]
tricky_summary[display_cols].style.apply(_highlight, axis=1)


## 9. Evaluación — precisión por tipo de frase y matriz de confusión

¿Dónde se equivoca NLU? Agregamos `correct` por `category` para responder a *qué tipo de retórica* le cuesta más al modelo. La matriz de confusión muestra el patrón global.


In [ ]:
if 'correct' in tricky_summary.columns:
    accuracy_by_cat = (
        tricky_summary.groupby('category')['correct']
        .mean()
        .mul(100)
        .round(1)
        .sort_values()
    )

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

    # --- Precisión por categoría ---
    palette = sns.color_palette('viridis', len(accuracy_by_cat))
    bars = axes[0].barh(accuracy_by_cat.index, accuracy_by_cat.values, color=palette)
    axes[0].set_xlim(0, 105)
    axes[0].set_xlabel('Precisión (%)')
    axes[0].set_title('Precisión de NLU por tipo de frase')
    for bar, value in zip(bars, accuracy_by_cat.values):
        axes[0].text(value + 1.5, bar.get_y() + bar.get_height() / 2,
                     f'{value:.0f}%', va='center', fontsize=9)

    # --- Matriz de confusión ---
    classes = ['negative', 'neutral', 'positive']
    cm = pd.crosstab(
        tricky_summary['expected'],
        tricky_summary['sentiment_label'],
        rownames=['esperado'], colnames=['predicho'],
    ).reindex(index=classes, columns=classes, fill_value=0)

    sns.heatmap(cm, annot=True, fmt='d', cmap='mako', ax=axes[1],
                cbar=False, linewidths=.5, linecolor='white')
    axes[1].set_title('Matriz de confusión — corpus adversarial')

    plt.tight_layout()
    plt.show()

    overall = tricky_summary['correct'].mean() * 100
    print(f'\nPrecisión global sobre {len(tricky_summary)} frases adversariales: {overall:.1f}%')
    print('Las categorías más bajas son donde NLU necesita un modelo de apoyo (p.ej. un transformer).')
else:
    print('No se generaron etiquetas comparables.')


## 10. Aplicar NLU a tickets reales de soporte

Cargamos una muestra del CSV `customer_support_tickets.csv` que alimenta la app **VoxCustomer** (`semana-04/voxcustomer/data/`). El loader probará varias rutas relativas; si lo ejecutas en Colab y no encuentra el archivo, súbelo con el panel de archivos y se detectará automáticamente.


In [ ]:
CANDIDATE_PATHS = [
    Path('../voxcustomer/data/customer_support_tickets.csv'),
    Path('../../semana-04/voxcustomer/data/customer_support_tickets.csv'),
    Path('voxcustomer/data/customer_support_tickets.csv'),
    Path('/content/customer_support_tickets.csv'),
]

def load_real_tickets(sample_size: int = 12, seed: int = 7) -> pd.DataFrame:
    for path in CANDIDATE_PATHS:
        if path.exists():
            print(f'Tickets cargados desde {path}')
            raw = pd.read_csv(path).dropna(subset=['Ticket Description']).copy()
            raw['Ticket Description'] = raw['Ticket Description'].str.replace(
                '{product_purchased}', 'product', regex=False
            )
            sampled = raw.sample(n=min(sample_size, len(raw)), random_state=seed)
            return pd.DataFrame({
                'id': 'ticket_' + sampled['Ticket ID'].astype(str).values,
                'source': sampled['Ticket Type'].fillna('unknown').values,
                'category': sampled.get('Ticket Priority', pd.Series(['unknown'] * len(sampled))).values,
                'language': 'en',
                'text': sampled['Ticket Description'].values,
            })
    print('No se encontró el CSV. Sube `customer_support_tickets.csv` manualmente en Colab.')
    return pd.DataFrame(columns=['id', 'source', 'category', 'language', 'text'])

real_tickets_df = load_real_tickets(sample_size=12)
real_tickets_df[['id', 'source', 'category', 'text']].head(12)


In [ ]:
if not real_tickets_df.empty:
    real_summary = analyze_dataset(real_tickets_df, client=nlu_client, expected_column=None)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

    # --- Distribución de sentimiento ---
    counts = real_summary['sentiment_label'].value_counts().reindex(
        ['negative', 'neutral', 'positive'], fill_value=0
    )
    sentiment_palette = {'negative': '#E74C3C', 'neutral': '#95A5A6', 'positive': '#27AE60'}
    bars = axes[0].bar(counts.index, counts.values,
                       color=[sentiment_palette[c] for c in counts.index])
    axes[0].set_title(f'Sentimiento en {len(real_summary)} tickets reales')
    axes[0].set_ylabel('Tickets')
    for bar, value in zip(bars, counts.values):
        axes[0].text(bar.get_x() + bar.get_width() / 2, value + 0.1,
                     str(value), ha='center', fontsize=10)

    # --- Mezcla emocional dominante ---
    emotion_counts = real_summary['dominant_emotion'].value_counts()
    if not emotion_counts.empty:
        axes[1].pie(
            emotion_counts.values, labels=emotion_counts.index,
            autopct='%1.0f%%', startangle=90,
            colors=sns.color_palette('Set2', len(emotion_counts)),
            wedgeprops={'edgecolor': 'white', 'linewidth': 1.5},
        )
        axes[1].set_title('Emoción dominante por ticket')
    else:
        axes[1].axis('off')

    plt.tight_layout()
    plt.show()

    print('\nTickets más negativos del lote:')
    most_negative = (
        real_summary.sort_values('sentiment_score').head(3)
        [['id', 'sentiment_label', 'sentiment_score', 'dominant_emotion', 'snippet']]
    )
    display(most_negative)
else:
    print('Sin tickets cargados — sección omitida.')


## 11. Próximos pasos sugeridos

- **Amplía el corpus adversarial** con frases de tu propio dominio (encuestas NPS, tickets de tu empresa, reseñas de tienda) y vuelve a calcular la precisión por categoría.
- **Compara NLU contra un segundo modelo** — por ejemplo, el TF-IDF + Transformer de la app **VoxCustomer** (`semana-04/voxcustomer/pages/2_Model_Evaluation.py`) — y construye una rutina de *ensemble* que solo escale a humano cuando los dos modelos discrepen.
- **Multiidioma** — el cliente acepta `language='es'`, `'fr'`, `'de'`, `'pt'`, etc. Las dos frases en español de este cuaderno son un buen punto de partida. Recuerda que `EmotionOptions` solo funciona en inglés.
- **Expande `Features`** con `EntitiesOptions`, `RelationsOptions`, `SyntaxOptions` o `ConceptsOptions` para enriquecer el análisis con grafos de entidades o sintaxis.
- **Exporta tus resultados** — `tricky_summary.to_csv('eval_adversarial.csv', index=False)` y `real_summary.to_csv('tickets_analizados.csv', index=False)` para integrarlos en reportes o dashboards.
- **Protege tus credenciales** — usa *Colab Secrets*, GitHub Actions secrets o IBM Cloud Secret Manager. **Nunca** incrustes la API key en el cuaderno.
